# Hand2Robot 2.0 — WiLoR + depth model, on Colab, with videos

Runs the pipeline on **our** clips and saves everything — numbers **and videos** — to Google Drive. Nothing here touches the live servers.

The pipeline is **one** way now: **WiLoR** for the hand and the **MoGe-2 depth model** for the lens, the table and the hand's size, **on the GPU only** (no CPU fallback: without a GPU a clip fails).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Put the clips in Drive: **My Drive / hand2robot2 / clips /** (`.mp4`; with its `.mp4.camera.json` beside it if the new app recorded it). Optional `hand2robot2/truth.csv`: `clip,object_width_cm`.

**Output, per clip, in My Drive / hand2robot2 / results /**
- `<clip>/videos/hand_overlay.mp4` — the tracked hand on the clip, with how its distances were made
- `<clip>/videos/robot_sim.mp4` — the SO-101 in MuJoCo replaying it (red dot: where the hand was)
- `<clip>/videos/two_view.mp4` — both side by side
- `summary.csv` — the numbers; the hand and robot files of every run beside the videos

Run the cells **one by one**, top to bottom.

In [ ]:
#@title 1 · Settings
CODE_REPO = 'https://github.com/Aravind-0808/pipeline-colab-tmp.git'   # temporary, public; branch main
USE_DRIVE = True        # False in VS Code Colab (Drive cannot be mounted there): clips in /content/hand2robot2/clips
HANDS = 2               # 2 for a two-hand task (the cup handover), 1 for one hand
MAKE_VIDEOS = True
# WiLoR's checkpoints are old-style pickles from the warmshao/WiLoR-mini Hugging Face
# repo, which PyTorch refuses by default because a pickle can run code when loaded.
# Running this notebook accepts that, on this throw-away Colab runtime only.

import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ROOT = '/content/drive/MyDrive/hand2robot2'
    except Exception as e:
        print('Drive not available here (', e, ') — using /content/hand2robot2')
        ROOT = '/content/hand2robot2'
else:
    ROOT = '/content/hand2robot2'
os.makedirs(f'{ROOT}/clips', exist_ok=True)
print('ROOT =', ROOT, '| clips:', sorted(f for f in os.listdir(f'{ROOT}/clips') if f.endswith('.mp4')))
import subprocess
import shutil
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if gpu is None or gpu.returncode != 0:
    raise SystemExit('No GPU on this runtime. Runtime → Change runtime type → T4 GPU → Save, then run '
                     'again from cell 1. If T4 is greyed out or you get a CPU runtime anyway, Colab\'s '
                     'free GPU time for today is used up: try again later (or Colab Pro).')
print('GPU:', gpu.stdout.strip())

In [ ]:
#@title 2 · Code and packages (5–10 min the first time; re-run to pick up a newer push)
import os
if not os.path.isdir('/content/pipeline/.git'):
    !git clone -q {CODE_REPO} /content/pipeline
else:
    !git -C /content/pipeline pull -q
%cd /content/pipeline
!git log --oneline -1
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y ffmpeg libegl1 libgles2 > /dev/null 2>&1
# PyTorch pinned to the CUDA 12.6 build Colab's driver runs: a newer lerobot or
# torchcodec otherwise pulls PyTorch 2.11 for CUDA 13, whose cuSOLVER fails on
# Colab's driver (the depth model's lens estimate: CUSOLVER_STATUS_INTERNAL_ERROR).
# lerobot pinned too: without it pip walks back through releases that lack [dataset].
# torchaudio with it: Colab's own is built for its newer PyTorch, and a dataset build
# that loads it fails (undefined symbol: torch_library_impl).
TORCH = ['torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1']
!pip install -q --index-url https://download.pytorch.org/whl/cu126 {' '.join(TORCH)}
!pip install -q -e ".[ml]" "lerobot[dataset]==0.6.1" mujoco {' '.join(TORCH)} --extra-index-url https://download.pytorch.org/whl/cu126
!python -c "import torch, torchaudio; print('PyTorch', torch.__version__, '| torchaudio', torchaudio.__version__, '| CUDA', torch.version.cuda); assert torch.__version__.startswith('2.7.1') and torchaudio.__version__.startswith('2.7.1')"
!pip -q uninstall -y opencv-contrib-python opencv-python opencv-python-headless
!pip -q install "opencv-contrib-python-headless==5.0.0.93"
!sh scripts/install_perception.sh depth
!python scripts/download_retarget_weights.py --dir ./models
!python scripts/download_perception_weights.py --dir ./models depth
# The SO-101 for the MuJoCo videos: the official model, the same robot as models/so101.
if not os.path.isdir('/content/so101_sim/Simulation/SO101'):
    !git clone -q --filter=blob:none --no-checkout https://github.com/TheRobotStudio/SO-ARM100.git /content/so101_sim
    !cd /content/so101_sim && git sparse-checkout set --no-cone Simulation/SO101 && git checkout -q a758567c3978dfeefe282ede0500085a48fe8f78
!python -c "import cv2, mediapipe, scipy, moge, mujoco, dfpipeline.worker; print('imports OK')"

In [ ]:
#@title 3 · WiLoR
# chumpy: a small stand-in, written below. The real chumpy no longer builds on
# Python 3.12+ / numpy 2, and WiLoR needs it only to read MANO's pickle (two classes).
import os, textwrap
d = '/content/chumpy_shim/chumpy'; os.makedirs(d, exist_ok=True)
open(f'{d}/__init__.py', 'w').write('from chumpy.ch import Ch\n')
open(f'{d}/ch.py', 'w').write(textwrap.dedent('''
    import numpy as np
    class Ch:
        # smplx reads .shape and slices before it converts with .r, so this
        # behaves as the array it holds.
        def __setstate__(self, state): self.__dict__.update(state)
        @property
        def r(self): return np.asarray(self.x)
        def __array__(self, dtype=None, copy=None):
            a = np.asarray(self.r)
            return a.astype(dtype) if dtype is not None else a
        @property
        def shape(self): return np.asarray(self.r).shape
        @property
        def ndim(self): return np.asarray(self.r).ndim
        @property
        def dtype(self): return np.asarray(self.r).dtype
        def __len__(self): return len(np.asarray(self.r))
        def __getitem__(self, i): return np.asarray(self.r)[i]
'''))
open(f'{d}/reordering.py', 'w').write(textwrap.dedent('''
    import numpy as np
    from chumpy.ch import Ch
    class Select(Ch):
        @property
        def r(self):
            v = np.asarray(self.a.r).ravel()[np.asarray(self.idxs)]
            s = getattr(self, "preferred_shape", None)
            return v.reshape(s) if s is not None else v
'''))
open('/content/chumpy_shim/pyproject.toml', 'w').write(textwrap.dedent('''
    [project]
    name = "chumpy"
    version = "0.0.0"
    dependencies = ["numpy"]
    [build-system]
    requires = ["setuptools>=61"]
    build-backend = "setuptools.build_meta"
    [tool.setuptools]
    packages = ["chumpy"]
'''))
!pip install -q --force-reinstall --no-deps /content/chumpy_shim
!pip install -q --no-deps "wilor-mini @ git+https://github.com/warmshao/WiLoR-mini.git@ebec42f94c389070cdd7dda6fd1bf0b4a659c960" "smplx==0.1.28" "ultralytics==8.1.34"
# --no-deps: thop and friends would otherwise upgrade PyTorch and pull a second
# OpenCV, which overwrites the headless one (cv2 then has no VideoCapture).
!pip install -q --no-deps timm einops roma py-cpuinfo thop
!pip install -q scikit-image torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --extra-index-url https://download.pytorch.org/whl/cu126
# Whatever came in, one OpenCV again: the headless contrib build MediaPipe takes.
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip -q install --force-reinstall --no-deps "opencv-contrib-python-headless==5.0.0.93"
!python -c "import cv2; assert hasattr(cv2, 'VideoCapture'); print('OpenCV OK', cv2.__version__)"
!python scripts/download_perception_weights.py --dir ./models wilor
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

In [ ]:
#@title 4 · Do the models run here? (device, seconds a frame, GPU memory)
import os
os.environ.update({'RETARGET_MODELS_DIR': './models', 'GPU_MEMORY_FRACTION': '0.9', 'MUJOCO_GL': 'egl'})
!python scripts/gpu_smoke.py

In [ ]:
#@title 5 · The runs (each clip in a process of its own; a few minutes each)
# A process per clip (research/run_one.py): the models in one notebook process ran
# Colab out of memory and the kernel died. Results land in Drive as they finish.
import csv, json, os, subprocess, sys
from pathlib import Path

clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
truth = {}
if Path(ROOT, 'truth.csv').exists():
    truth = {r['clip']: r['object_width_cm'] for r in csv.DictReader(open(Path(ROOT, 'truth.csv')))}
out_root = Path(ROOT, 'results'); out_root.mkdir(exist_ok=True)
env = {**os.environ, 'RETARGET_MODELS_DIR': '/content/pipeline/models', 'GPU_MEMORY_FRACTION': '0.9',
       'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'}
print(len(clips), 'clips | WiLoR + depth model | hands:', HANDS)

rows = []
for clip in clips:
    print(f'running {clip.name} …', flush=True)
    run = subprocess.run([sys.executable, 'research/run_one.py', str(clip), str(HANDS), str(out_root),
                          str(truth.get(clip.name, ''))],
                         cwd='/content/pipeline', env=env, capture_output=True, text=True)
    row_file = out_root / clip.stem / 'row.json'
    if row_file.exists():
        row = json.loads(row_file.read_text())
    else:   # the process died before writing its row (out of memory, a native crash)
        row = {'clip': clip.name, 'error': f'exit {run.returncode}: ' + (run.stderr or '')[-600:]}
    rows.append(row)
    status = 'ERROR ' + (row.get('error') or '')[-300:] if row.get('error') else \
        f"done: hand {row.get('hand_s')} s, lens {row.get('p_lens_source')}, robot error {row.get('position_error_median_m')} m"
    print('   ', status, flush=True)
json.dump(rows, open(out_root / 'rows.json', 'w'), indent=1, default=str)

In [ ]:
#@title 6 · The numbers
# After a restart, what cell 5 saved to Drive (results/rows.json) is read back:
# the runs are not repeated.
import json
from pathlib import Path
if 'rows' not in globals():
    out_root = Path(ROOT, 'results')
    rows = json.load(open(out_root / 'rows.json'))
    print('read back', len(rows), 'results')
import pandas as pd
df = pd.DataFrame(rows)
cols = ['clip', 'hand_ok', 'hand_s', 'p_lens_source', 'p_hfov_deg', 'p_table_source', 'table_distance_m',
        'p_hand_scale', 'p_hand_scale_spread', 'p_table_up_angle_deg',
        'object_width_cm', 'truth_width_cm', 'hand_coverage', 'robot_ok', 'reachable_fraction',
        'position_error_median_m', 'slowdown', 'notes']
table = df[[c for c in cols if c in df.columns]].copy()
for c in ('hand_s', 'table_distance_m', 'object_width_cm', 'truth_width_cm', 'reachable_fraction',
          'position_error_median_m', 'slowdown'):
    if c in table:
        table[c] = pd.to_numeric(table[c], errors='coerce')
if 'truth_width_cm' in table and table['truth_width_cm'].notna().any() and 'object_width_cm' in table:
    table['width_error_cm'] = (table['object_width_cm'] - table['truth_width_cm']).abs()
    print('median object-width error:', round(table['width_error_cm'].median(), 1), 'cm')
headline = [c for c in ('hand_s', 'reachable_fraction', 'position_error_median_m', 'slowdown') if c in table]
if headline:
    print(table[headline].median().rename('median over the clips'))
if 'error' in df:
    for _, r in df[df['error'].notna()].iterrows():
        print('ERROR in', r['clip'], '→', str(r['error'])[-300:])
table.to_csv(out_root / 'summary.csv', index=False)
table

In [ ]:
#@title 7 · The videos (to Drive: results/<clip>/videos/)
# Each clip's videos are made in a process of their own, with a time limit, on the
# local disk, then copied to Drive: on one runtime MuJoCo wrote nothing in an hour,
# and Drive is slow for a video written in small pieces. MuJoCo is tried first on ten
# frames; where it hangs or crawls, the robot is drawn as a stick figure instead.
import json, os, shutil, subprocess, sys
from pathlib import Path
if 'rows' not in globals():
    out_root = Path(ROOT, 'results')
    rows = json.load(open(out_root / 'rows.json'))
clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
env = {**os.environ, 'MUJOCO_GL': 'egl'}
SO101 = '/content/so101_sim/Simulation/SO101'
if MAKE_VIDEOS:
    try:
        probe = subprocess.run([sys.executable, 'research/sim_probe.py', SO101], cwd='/content/pipeline',
                               env=env, capture_output=True, text=True, timeout=120)
        ms = float(probe.stdout.strip().splitlines()[-1]) if probe.returncode == 0 else None
    except (subprocess.TimeoutExpired, ValueError, IndexError):
        ms = None
    sim = SO101 if ms is not None and ms < 200 else '-'
    print(f'MuJoCo: {ms:.0f} ms a frame' if ms is not None else 'MuJoCo: does not render here',
          '→ 3D robot' if sim != '-' else '→ stick view')
    for clip in clips:
        run = out_root / clip.stem
        if not (run / 'hand' / 'hand_stage.json').exists():
            print(clip.name, ': no run to show (see cell 6)')
            continue
        local = Path('/content/videos') / clip.stem
        shutil.rmtree(local, ignore_errors=True)
        print(f'videos for {clip.name} …', flush=True)
        try:
            made = subprocess.run([sys.executable, 'research/render_one.py', str(clip), str(run), str(local), sim],
                                  cwd='/content/pipeline', env=env, capture_output=True, text=True, timeout=1800)
            ok = made.returncode == 0
            print('   ', made.stdout.strip() if ok else 'failed: ' + (made.stderr or '')[-400:])
        except subprocess.TimeoutExpired:
            ok = False
            print('    stopped after 30 min')
        if ok:
            shutil.copytree(local, run / 'videos', dirs_exist_ok=True)
            print('    in Drive:', run / 'videos')

In [ ]:
#@title 8 · Watch one here (the first clip's hand and robot, side by side)
from IPython.display import Video
first = sorted(Path('/content/videos').glob('*/two_view.mp4'))
Video(str(first[0]), embed=True, width=960) if first else print('no video yet')

**Send back** `results/summary.csv` and tell me what the videos show — does the robot follow the hand? When you are done: **Runtime → Disconnect and delete runtime**.